# 検索広告の見出しを LLM で複数作り、文字数やキーワードを自動でチェックして選ぶ

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_llm/ad_text_generation.ipynb)

NLP クックブック 12-14 ／ 大規模言語モデル (LLM)

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

商品やサービスの説明から、検索連動型広告の見出し（半角 30 文字・全角 15 文字まで）を LLM で何本も作り、文字数の超過、重複、キーワードの有無、説明にない数値や誇大な表現を自動でチェックして良いものを選びたい。公開データで、人が作った見出しとの類似度による自動評価も試したい。

In [1]:
!pip install -q rouge-score sacrebleu fugashi unidic-lite

In [2]:
import re, unicodedata
import numpy as np
import pandas as pd
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)
name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"       # MIT ライセンス
tokenizer = AutoTokenizer.from_pretrained(name, padding_side="left")
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま)
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()

RULES = ("検索連動型広告の見出しを 1 つ作ってください。\n"
         "- 全角 15 文字 (半角 30 文字) 以内にする\n- 検索キーワードの語をなるべく含める\n"
         "- 説明に書かれていない数値、実績、「No.1」「最安」などの表現は使わない\n- 見出しだけを出力する")

def prompt(kw, desc, shots=()):
    msgs = []
    for s_kw, s_desc, s_title in shots:       # 例は「ユーザーの依頼 → アシスタントの見出し」の対話として前に置く
        msgs += [{"role": "user", "content": f"{RULES}\n\n検索キーワード: {s_kw}\n説明: {s_desc}"}, {"role": "assistant", "content": s_title}]
    msgs.append({"role": "user", "content": f"{RULES}\n\n検索キーワード: {kw}\n説明: {desc}"})
    return tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)

def clean(text):
    """1 行目だけを取り、前後の記号 (かぎかっこ、箇条書きの印、番号) を除く"""
    line = text.strip().split("\n")[0].strip()
    line = re.sub(r"^(?:[-・*●]|\d+[.．)）])\s*", "", line)
    line = re.sub(r"[（(\[［【][^）)\]］】]*文字[^）)\]］】]*[）)\]］】]", "", line).strip()   # 「（全角15文字）」のような注記を除く
    return line.strip("「」『』\"' 　")

@torch.no_grad()
def generate(prompts, n=1, bs=32, max_new_tokens=32):
    """プロンプトごとに n 本の見出しを返す。n=1 なら貪欲法、n>1 ならサンプリング (左パディングのバッチ生成は、この章の「複数のプロンプトをまとめてバッチ生成する」と同じ)"""
    outs, step = [], max(1, bs // n)
    for i in range(0, len(prompts), step):
        enc = tokenizer(prompts[i:i + step], return_tensors="pt", padding=True, add_special_tokens=False).to(llm.device)
        sample = dict(do_sample=True, temperature=1.0, top_p=0.95) if n > 1 else dict(do_sample=False)
        out = llm.generate(**enc, max_new_tokens=max_new_tokens, num_return_sequences=n, pad_token_id=tokenizer.pad_token_id, **sample)
        outs += [clean(t) for t in tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)]
    return [outs[j * n:(j + 1) * n] for j in range(len(prompts))]

def width(s):
    """全角 (East Asian Width が F か W) を 2、それ以外を 1 と数える。Google 広告の見出しの上限は 30"""
    return sum(2 if unicodedata.east_asian_width(c) in "FW" else 1 for c in s)

def shorten(cands, limit=30, rounds=3):
    """上限を超えた見出しだけを、今の長さを伝えて LLM に短く言い換えさせる (貪欲法)。最大 rounds 回くり返す"""
    cands = list(cands)
    for _ in range(rounds):
        long = [i for i, c in enumerate(cands) if width(c) > limit]
        if not long:
            break
        ask = [tokenizer.apply_chat_template([{"role": "user", "content":
                   f"次の広告の見出しは全角に換算して {width(cands[i]) / 2:g} 文字あり、上限の {limit / 2:g} 文字を超えています。"
                   f"意味を保ったまま {limit / 2:g} 文字以内に短くしてください。新しい情報は加えず、短くした見出しだけを出力してください。\n\n{cands[i]}"}],
                   add_generation_prompt=True, tokenize=False) for i in long]
        for i, (c,) in zip(long, generate(ask)):
            cands[i] = c
    return cands

In [3]:
# チェックの関数: キーワード、説明にない数値、誇大になりやすい表現、重複 (文字幅の width は前のセル)
norm = lambda s: unicodedata.normalize("NFKC", s).lower().replace(",", "")
def kw_coverage(text, kw):
    """検索キーワードの語 (空白で区切る) のうち、見出しに含まれる割合"""
    words = norm(kw).split()
    return sum(w in norm(text) for w in words) / len(words) if words else 0.0

def unsupported_numbers(text, source):
    """見出しに出てくる数値のうち、説明にないもの"""
    return [x for x in re.findall(r"\d+(?:\.\d+)?", norm(text)) if x not in re.findall(r"\d+(?:\.\d+)?", norm(source))]

HYPE = r"no\.?1|ナンバーワン|日本一|業界一|世界一|日本初|業界初|世界初|最安|最高|最大|最短|唯一|絶対|必ず|完全|100%|永久|誰でも"
def hype_words(text, source):
    """誇大になりやすい表現のうち、説明に出てこないもの (使ってよいかは根拠しだいなので、人が確かめる印として使う)"""
    return [w for w in re.findall(HYPE, norm(text)) if w not in norm(source)]

def bigrams(s):
    s = norm(re.sub(r"\s", "", s))
    return {s[i:i + 2] for i in range(len(s) - 1)} or {s}
def similar(a, b):
    """文字 2-gram のジャッカード係数"""
    return len(bigrams(a) & bigrams(b)) / len(bigrams(a) | bigrams(b))

def check(cands, kw, source, limit=30):
    rows = []
    for c in cands:
        rows.append({"見出し": c, "幅": width(c), "幅OK": 0 < width(c) <= limit, "KW": round(kw_coverage(c, kw), 2),
                     "説明にない数値": unsupported_numbers(c, source), "誇大表現": hype_words(c, source)})
    df = pd.DataFrame(rows)
    df["要確認"] = df["説明にない数値"].map(bool) | df["誇大表現"].map(bool)
    # 合格した候補どうしの類似度の平均 (多くの候補と内容が共通する、無難な候補ほど高い。最小ベイズリスク復号の考え方)
    ok = df["幅OK"] & ~df["要確認"]
    df["合意"] = [np.mean([similar(c, d) for j, d in enumerate(df["見出し"]) if j != i and ok[j]] or [0]) for i, c in enumerate(df["見出し"])]
    return df

def select(df, k=3, max_sim=0.5):
    """合格した候補を KW の割合 → 合意の順に並べ、すでに選んだものと似すぎるもの (重複) を飛ばして k 本選ぶ"""
    picked = []
    for _, r in df[df["幅OK"] & ~df["要確認"]].sort_values(["KW", "合意"], ascending=False).iterrows():
        if all(similar(r["見出し"], p) < max_sim for p in picked):
            picked.append(r["見出し"])
        if len(picked) == k:
            break
    return picked

# 架空のサービスで試す
kw = "オンライン英会話 初心者"
desc = ("オンライン英会話「ことばラボ」。講師は全員日本語で説明できるので、英語が初めての方も安心。"
        "1 回 25 分、月 8 回で月額 6,980 円 (税込)。無料体験レッスンは 2 回まで。朝 6 時から深夜 1 時まで予約できる。")
torch.manual_seed(0)
cands = generate([prompt(kw, desc)], n=16)[0]
pd.set_option("display.width", 200); pd.set_option("display.unicode.east_asian_width", True)
print("生成した候補の幅:", [width(c) for c in cands], cands[:3])
table = check(shorten(cands), kw, desc)      # 上限を超えたものを短くさせてから、もう一度チェックする
print(table.drop(columns="合意").to_string())
print("選んだ見出し:", select(table))

生成した候補の幅: [67, 61, 51, 51, 61, 76, 60, 81, 58, 79, 63, 64, 82, 68, 48, 61] ['初心者向けオンライン英会話なら『ことばラボ』 - 日本語サポートで安心', '初心者も安心、日本語サポート付きオンライン英会話 - ことばラボ', '初心者歓迎、日本語OKのオンライン英会話 - ことばラボ']
                                                             見出し  幅   幅OK   KW 説明にない数値 誇大表現  要確認
0        初心者向けオンライン英会話『ことばラボ』日本語サポート付き  58  False  1.0             []       []   False
1                   初心者も安心、日本語サポート英会話 - ことばラボ  47  False  0.5             []       []   False
2                         初心者歓迎、オンライン英会話 - ことばラボ  41  False  1.0             []       []   False
3               初心者歓迎、日本語対応オンライン英会話 - ことばラボ  51  False  1.0             []       []   False
4   英語初心者も安心、日本語サポートのオンライン英会話 - ことばラボ  63  False  1.0             []       []   False
5         初心者歓迎、オンライン英会話ならことばラボ（月額6,980円）  57  False  1.0             []       []   False
6                初心者安心、オンライン英会話 ことばラボ - 講師常駐  50  False  1.0             []       []   False
7                             英語初心者歓迎、ことばラボ月額6,980円  37  False  0.5             [

In [4]:
# CAMERA (CyberAgent): 実際の検索広告から作った日本語の広告文生成のデータセット (Mita et al., 2024)。CC BY-NC-SA 4.0 (非商用に限る)
# 入力は検索キーワード (kw) と LP (広告のリンク先のページ) の説明文 (lp_meta_description)。
# テストデータは 1 件に 4 本の参照の見出しがある (title_org は実際の広告、title_ne1〜3 は全角 15 文字以内で人が書き足したもの)
def camera(split):
    path = hf_hub_download("cyberagent/camera", f"without-lp-images/{split}-00000-of-00001.parquet", repo_type="dataset")
    return pd.read_parquet(path).drop(columns="parsed_full_text_annotation")

test = camera("test").sample(120, random_state=0).reset_index(drop=True)      # 時間の都合で 872 件から 120 件
# few-shot の例は架空のものを自分で書く (CAMERA の学習・開発データには、キーワードと LP の内容が合わない組も混じるため)
shots = [("家計簿 アプリ 無料", "レシートを撮るだけで自動で記帳できる家計簿アプリ「マネノート」。基本機能は無料。銀行口座やカードと連携でき、グラフで支出を確認できる。",
          "レシート撮影で家計簿が自動に"),
         ("渋谷 ヨガ 体験", "渋谷駅から徒歩 3 分のヨガスタジオ「アサナ渋谷」。初心者向けのクラスが中心で、体験レッスンは 1,000 円。マットやウェアは無料で借りられる。",
          "渋谷駅3分のヨガ 体験1000円")]
print([(t, width(t)) for *_, t in shots])

# 貪欲法で 1 本と、サンプリングで 8 本ずつ。上限を超えた候補は短くさせる (Colab の T4 で 10 分程度)
prompts = [prompt(k, d, shots) for k, d in zip(test["kw"], test["lp_meta_description"])]
greedy = [c[0] for c in generate(prompts)]
torch.manual_seed(0)
samples = generate(prompts, n=8)
flat = shorten([c for cs in samples for c in cs])                  # 全候補をまとめて短くさせる (バッチにするため)
shortened = [flat[j * 8:(j + 1) * 8] for j in range(len(samples))]
def pick(cands_list):
    checked = [check(c, k, d) for c, k, d in zip(cands_list, test["kw"], test["lp_meta_description"])]
    # 合格がなければ、幅OK → KW の順で最上位のもの (実際には人が書くか、生成し直す)
    return checked, [(select(t, k=1) or [t.sort_values(["幅OK", "KW"], ascending=False)["見出し"].iloc[0]])[0] for t in checked]
checked0, picked0 = pick(samples)
checked, picked = pick(shortened)
print(f"GPU メモリ最大 {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
for i in range(4):
    print(f"KW: {test['kw'][i]} / 実際の広告: {test['title_org'][i]} / 貪欲: {greedy[i]} / 選択: {picked[i]}")

[('レシート撮影で家計簿が自動に', 28), ('渋谷駅3分のヨガ 体験1000円', 26)]
GPU メモリ最大 10.8 GB
KW: 福山市 土地 売却 / 実際の広告: 福山の土地を売却するなら / 貪欲: 福山市の土地売却、まずは査定【SUUMO】 / 選択: 福山市の土地売却、3社査定比較
KW: は な さくせい めい ほけん / 実際の広告: がんでも申し込める死亡保険 / 貪欲: 日本初のがん経験者向け保険、リニューアル！ / 選択: がん経験者向け保険、富士の保証
KW: 東京 銀座 バイト / 実際の広告: 【池袋】チャットレディ求人募集 / 貪欲: 銀座バイトで祝い金GET（東京） / 選択: 銀座バイトで祝い金GET（東京）
KW: マーキーズ マルイ / 実際の広告: 冬の最新スタイルも勢揃い / 貪欲: マーキーズ秋冬新作、マルイ通販で解決 / 選択: マーキーズ秋冬新作 通販サイト


In [5]:
# 8 本の候補のチェック結果の集計: 幅の超過、重複、要確認の割合
for label, ch in [("短くする前", checked0), ("短くした後", checked)]:
    a = pd.concat(ch, ignore_index=True)
    print(f"{label}: 幅の超過 {1 - a['幅OK'].mean():.1%}  要確認 {a['要確認'].mean():.1%}  1 本も合格しなかった入力 {np.mean([not select(t, k=1) for t in ch]):.1%}")
allc = pd.concat(checked, ignore_index=True)      # 短くした後の全候補
distinct = np.mean([len(set(map(norm, c))) / len(c) for c in samples])
near_dup = np.mean([np.mean([similar(a, b) >= 0.5 for i, a in enumerate(c) for b in c[i + 1:]]) for c in samples])
print(f"短くする前の {len(allc)} 本: 完全に同じ候補を除くと残る割合 {distinct:.1%}  "
      f"よく似た組 (2-gram の類似度 0.5 以上) の割合 {near_dup:.1%}")
print(f"短くした後: 説明にない数値 {allc['説明にない数値'].map(bool).mean():.1%}  誇大表現 {allc['誇大表現'].map(bool).mean():.1%}")
print("短くした例:", [(a, b) for a, b in zip(samples[0], shortened[0]) if a != b][:3])
print("説明にない数値を含む候補の例:", allc.loc[allc["説明にない数値"].map(bool), "見出し"].head(5).tolist())
print("誇大表現を含む候補の例:", allc.loc[allc["誇大表現"].map(bool), "見出し"].head(5).tolist())

短くする前: 幅の超過 80.1%  要確認 1.9%  1 本も合格しなかった入力 41.7%
短くした後: 幅の超過 56.5%  要確認 2.0%  1 本も合格しなかった入力 11.7%
短くする前の 960 本: 完全に同じ候補を除くと残る割合 93.0%  よく似た組 (2-gram の類似度 0.5 以上) の割合 27.7%
短くした後: 説明にない数値 2.0%  誇大表現 0.2%
短くした例: [('福山市土地売却、査定額比較で高く売るならSUUMO', '福山市の土地売却、SUUMOで査定比較し高値売却'), ('福山市の土地売却、査定は3社以上で比較', '福山市の土地売却、査定は「比較」で賢く'), ('福山市の土地売却、3社査定で比較', '福山市の土地売却、3社査定比較')]
説明にない数値を含む候補の例: ['銀座バイトで祝金3万GET（東京）', 'オリックス生命の死亡保障、相談・契約は15文字以内で: "死亡保障相談・契約（オリックス生命）', 'アクサダイレクト自動車保険見積り、割引最大15,000円', '通信で学べる名門会の家庭教師サービス - 合格実績豊富（15文字以内調整版: 「名門会家庭教師、通信で合格実績', 'Udemyオンライン学習: 21万コース、5,700万人が受講']
誇大表現を含む候補の例: ['タウンワークで求人探し - 人気No.1', 'タウンワーク 求人探しはNo.1！']


In [6]:
# 参照との自動評価: 人が書いた title_ne1〜3 を参照にし、実際の広告 (title_org) も同じ参照で採点して人の目安にする
from fugashi import Tagger
from rouge_score import rouge_scorer
from sacrebleu.metrics import CHRF

tagger = Tagger()
class Tok:
    def tokenize(self, text):
        return [w.surface for w in tagger(text)]
rouge = rouge_scorer.RougeScorer(["rouge1"], tokenizer=Tok())
chrf = CHRF()
refs = test[["title_ne1", "title_ne2", "title_ne3"]].values.tolist()

def scores(hyps):
    return pd.DataFrame({
        "ROUGE-1": [rouge.score_multi(r, h)["rouge1"].fmeasure for r, h in zip(refs, hyps)],      # 参照ごとの F の最大
        "chrF": [chrf.sentence_score(h, r).score / 100 for r, h in zip(refs, hyps)],
        "幅OK": [0 < width(h) <= 30 for h in hyps],
        "KW": [kw_coverage(h, k) for h, k in zip(hyps, test["kw"])],
        "要確認": [bool(unsupported_numbers(h, d) or hype_words(h, d)) for h, d in zip(hyps, test["lp_meta_description"])]})

systems = {"貪欲法 1 本": greedy, "サンプリング 1 本目": [c[0] for c in samples], "8 本からチェックして選ぶ": picked0,
           "8 本を短くしてから選ぶ": picked, "実際の広告 (title_org)": test["title_org"].tolist()}
results = {s: scores(h) for s, h in systems.items()}
print(pd.DataFrame({s: r.mean() for s, r in results.items()}).T.round(3).to_string())

rng = np.random.default_rng(0)
def diff_ci(a, b, col, n_boot=2000):
    """同じ入力で比べる対応のあるブートストラップで、平均の差の 95% 信頼区間"""
    d = (results[a][col] - results[b][col].astype(float)).to_numpy(dtype=float)
    boot = d[rng.integers(0, len(d), (n_boot, len(d)))].mean(axis=1)
    print(f"{a} − {b} ({col}): {d.mean():+.3f} [{np.percentile(boot, 2.5):+.3f}, {np.percentile(boot, 97.5):+.3f}]")

for col in ["ROUGE-1", "chrF", "KW"]:
    diff_ci("8 本を短くしてから選ぶ", "貪欲法 1 本", col)
diff_ci("8 本を短くしてから選ぶ", "実際の広告 (title_org)", "ROUGE-1")

                          ROUGE-1   chrF   幅OK     KW  要確認
貪欲法 1 本                 0.222  0.113  0.283  0.808   0.017
サンプリング 1 本目         0.201  0.106  0.192  0.796   0.017
8 本からチェックして選ぶ    0.207  0.103  0.592  0.859   0.008
8 本を短くしてから選ぶ      0.210  0.101  0.892  0.811   0.008
実際の広告 (title_org)      0.184  0.104  0.942  0.320   0.075
8 本を短くしてから選ぶ − 貪欲法 1 本 (ROUGE-1): -0.011 [-0.034, +0.013]
8 本を短くしてから選ぶ − 貪欲法 1 本 (chrF): -0.013 [-0.027, -0.000]
8 本を短くしてから選ぶ − 貪欲法 1 本 (KW): +0.003 [-0.040, +0.046]
8 本を短くしてから選ぶ − 実際の広告 (title_org) (ROUGE-1): +0.026 [-0.007, +0.059]


- 見出しの文字数の上限は、Google 広告のヘルプでは半角 30 文字で、全角 1 文字を半角 2 文字と数える（全角だけなら 15 文字）。ここでは Unicode の East Asian Width が F（全角）か W（広い）の文字を 2 と数えた。「※」「…」のような幅が文脈で変わる文字 (A) を媒体がどう数えるかは確かめていないので、入稿の前に媒体の画面で確かめる。媒体や広告の種類ごとに上限が違うので、`limit` と `width` をそれに合わせる。
- このモデル (3B) では、「全角 15 文字以内」と指示しても文字数はほとんど守られなかった。架空のサービスの例では 16 本すべてが上限を超え（幅 48〜82）、今の長さを伝えて 3 回短くさせた後も 15 本が超えたままだった。CAMERA の 120 件でも、サンプリングした候補の 80.1%、貪欲法の見出しの 71.7% が超えた。最大 3 回短くさせても、上限に収まらない候補が 56.5% 残った。それでも候補を 8 本作っておけば、合格が 1 本もない入力は 41.7% から 11.7% に減り、選んだ見出しの 89.2% が上限に収まった（実際の広告は 94.2%）。文字数は生成のあとに必ずプログラムで数え、超えたものは捨てるか人が直す。上限で切り詰めると、文の途中で切れてしまう。
- 候補は、完全に同じものを除いても 93.0% 残ったが、組の 27.7% は文字 2-gram の類似度が 0.5 以上で、言い回しが少し違うだけのものが多い。`select` のように、すでに選んだ見出しと似すぎるものを飛ばすと、レスポンシブ検索広告のように複数の見出しを入稿するとき、訴求の違うものをそろえやすい。合格した候補の中では、キーワードの語の割合、ほかの候補との類似度の平均（多くの候補と共通する無難な内容ほど高い）の順で並べた。クリック率のような効果は、配信して確かめるしかない。
- 説明にない数値や「No.1」「最安」のような表現は、根拠がなければ景品表示法の優良誤認・有利誤認にあたるおそれがあり、広告主の信用も損なう。「説明にない数値を使わない」と指示しても、短くした後の候補の 2.0% が説明にない数値で、0.2% が誇大になりやすい表現で引っかかった。本当に説明と違う例には、説明の「インターネット割引最大20,000円」を「割引最大15,000円」と書いた候補や、指示の「15文字以内」を書き写した候補があった。一方で、ここでの判定は「説明の文字列に出てこない」という印にすぎず、説明の「30,000円」を「3万」、英語の説明の「213,000 courses and 57 million students」を「21万コース、5,700万人」と書いた候補のような表記の違いも引っかかる。「人気No.1」の 2 本は、説明の「ブランド認知度No.01」を言い換えたもので、表記が違うので文字列の照合では通らなかったが、「認知度」が「人気」に変わって根拠の範囲も変わっている。逆に説明にある「日本初」は通る（説明が正しいかは確かめない）。実際の広告 (title_org) でも 7.5% が引っかかった。LP の説明文 (meta description) はページの一部なので、人が書いた見出しには、ページのほかの部分にある数値も入る。引っかかったものは、広告主の資料と照らして人が確かめる。
- 参照との自動評価では、方法の違いがほとんど見えなかった。人が書いた 3 本の参照に対する ROUGE-1（参照ごとの F の最大）は、どの方法も 0.18〜0.22 で、短くしてから選んだ見出しと貪欲法の差は −0.011 [−0.034, +0.013]（対応のあるブートストラップの 95% 信頼区間。評価指標と統計的検定の章の「ブートストラップ法で評価値の信頼区間を求める」を参照）だった。chrF は −0.013 [−0.027, −0.000] とわずかに下がったが、信頼区間の端が 0 に接する程度である。実際の広告 (0.184) との差も +0.026 [−0.007, +0.059] で、人が書いた見出しも LLM と同じくらい参照と重ならない。広告文は同じ商品でも訴求の仕方がいくつもあるので、参照との重なりは良し悪しの目安にしかならない。CAMERA の論文 (Mita et al., 2024) も、BLEU-4、ROUGE-1、BERTScore のほかに、キーワードを含む割合と文字数の規定を守る割合を指標にし、人手評価との比較では、魅力 (attractiveness) の順位を既存の自動評価指標で再現するのは難しかったと報告している。記事の本文から見出しを生成して実際の見出しと比べる例は、要約と平易化の章の「記事の見出しを生成して実際の見出しと比べる」にある。
- キーワードの語を含む割合は、LLM の見出し (0.80〜0.86) のほうが実際の広告 (0.32) よりずっと高く、実際の広告はキーワードの語をそのまま含まないことが多い。ただし、CAMERA のキーワード (kw) は検索語なので、「は な さくせい めい ほけん」のように分かち書きが崩れたものや、LP や参照と内容が合わないもの（「東京 銀座 バイト」に対して実際の広告が池袋の求人）も混じるので、実際の広告の低さには、キーワードと広告が合わない組の影響も含まれる。件数も 120 件なので、細かな差は読み取らない。
- データは CAMERA（CyberAgent、CC BY-NC-SA 4.0）。非商用に限られ、改変したものも同じライセンスで公開する必要がある。商用の広告文生成のモデルの学習や評価には使えない。ここでは LP の画像を含まない without-lp-images のテストデータ（872 件）の parquet だけを読み込んでいる。モデルは sarashina2.2（SB Intuitions、MIT ライセンス）で、T4 と同じ float16 で実行したとき GPU メモリを最大 10.8 GB 使った。トークナイザーの修正については、この章の「オープンな LLM とチャットする（チャットテンプレート）」の注意書きを参照。